# 01 Relacionamentos e granularidade

Avaliar a integração entre vendas e canais e identificar relações relevantes para o modelo dimensional.

### Preparação das fontes

Disponibiliza os arquivos e as chaves necessárias às comparações.

In [0]:
from pyspark.sql import functions as F

LANDING = "/Volumes/workspace/beverage_sales_landing/source_files"
sales_raw = (
    spark.read.option("header", True).option("sep", "\t").option("inferSchema", False)
    .option("ignoreLeadingWhiteSpace", False).option("ignoreTrailingWhiteSpace", False)
    .option("mode", "FAILFAST").option("encoding", "UTF-16").option("multiLine", True)
    .csv(f"{LANDING}/abi_bus_case1_beverage_sales_20210726.csv")
)
channels_raw = (
    spark.read.option("header", True).option("sep", ",").option("inferSchema", False)
    .option("ignoreLeadingWhiteSpace", False).option("ignoreTrailingWhiteSpace", False)
    .option("mode", "FAILFAST").option("encoding", "UTF-8")
    .csv(f"{LANDING}/abi_bus_case1_beverage_channel_group_20210726.csv")
)
expected_sales = [
    "DATE", "CE_BRAND_FLVR", "BRAND_NM", "Btlr_Org_LVL_C_Desc", "CHNL_GROUP",
    "TRADE_CHNL_DESC", "PKG_CAT", "Pkg_Cat_Desc", "TSR_PCKG_NM",
    "$ Volume", "YEAR", "MONTH", "PERIOD",
]
expected_channels = ["TRADE_CHNL_DESC", "TRADE_GROUP_DESC", "TRADE_TYPE_DESC"]
if sales_raw.columns != expected_sales or channels_raw.columns != expected_channels:
    raise ValueError("Cabeçalho inesperado: conferir os arquivos antes de continuar.")


def normalized_key(column):
    value = F.upper(F.trim(column))
    return F.when(F.length(value) > 0, value)


sales = (
    sales_raw.withColumn("channel_key", normalized_key(F.col("TRADE_CHNL_DESC")))
    .withColumn("sales_amount", F.expr("try_cast(trim(`$ Volume`) AS decimal(28,6))"))
)
channels = channels_raw.withColumn("channel_key", normalized_key(F.col("TRADE_CHNL_DESC")))

## 1 Unicidade da classificação de canais

Verifica se cada canal possui uma única classificação. Chaves repetidas podem multiplicar vendas no cruzamento; classificações conflitantes exigem revisão.

In [0]:
channel_keys = channels.groupBy("channel_key").agg(
    F.count("*").alias("registros"),
    F.countDistinct("TRADE_CHNL_DESC").alias("descricoes_originais"),
    F.countDistinct(F.struct("TRADE_GROUP_DESC", "TRADE_TYPE_DESC")).alias("classificacoes"),
)
display(channel_keys.orderBy(F.desc("registros"), "channel_key"))
print("Chaves ausentes, repetidas ou com classificações conflitantes:")
display(channel_keys.filter(
    F.col("channel_key").isNull() | (F.col("registros") > 1) | (F.col("classificacoes") > 1)
))

channel_key,registros,descricoes_originais,classificacoes
ALL OTHER,1,1,1
AMUSEMENT/RECREATIONAL,1,1,1
AUTOMOTIVE SERVICES,1,1,1
BAR/TAVERN/LICENSED EST,1,1,1
CASH + CARRY RETAIL,1,1,1
CLUB STORE,1,1,1
COLLEGE/UNIVERSITY,1,1,1
CONVENIENCE STORE,1,1,1
DRUG STORE,1,1,1
GENERAL MERCHANDISER,1,1,1


Chaves ausentes, repetidas ou com classificações conflitantes:


channel_key,registros,descricoes_originais,classificacoes


## 2 Cobertura do relacionamento

Identifica vendas sem classificação e canais sem vendas. Compara a correspondência entre os valores originais e as chaves padronizadas.

In [0]:
unmatched_original = sales_raw.join(
    channels_raw.select("TRADE_CHNL_DESC").distinct(), ["TRADE_CHNL_DESC"], "left_anti"
)
unmatched_normalized = sales.join(
    channels.select("channel_key").distinct(), ["channel_key"], "left_anti"
)
unused_channels = channels.join(
    sales.select("channel_key").distinct(), ["channel_key"], "left_anti"
)
display(spark.createDataFrame(
    [
        ("vendas_sem_canal_no_join_original", unmatched_original.count()),
        ("vendas_sem_canal_apos_trim_upper", unmatched_normalized.count()),
        ("registros_de_canais_sem_vendas", unused_channels.count()),
    ],
    "verificacao string, registros long",
))
print("Canais de vendas sem correspondência original:")
display(unmatched_original.groupBy("TRADE_CHNL_DESC").count().orderBy(F.desc("count")))
print("Canais de vendas sem correspondência após normalização:")
display(unmatched_normalized.groupBy("TRADE_CHNL_DESC", "channel_key").count().orderBy(F.desc("count")))
print("Classificações sem vendas correspondentes:")
display(unused_channels)

verificacao,registros
vendas_sem_canal_no_join_original,0
vendas_sem_canal_apos_trim_upper,0
registros_de_canais_sem_vendas,0


Canais de vendas sem correspondência original:


TRADE_CHNL_DESC,count


Canais de vendas sem correspondência após normalização:


TRADE_CHNL_DESC,channel_key,count


Classificações sem vendas correspondentes:


channel_key,TRADE_CHNL_DESC,TRADE_GROUP_DESC,TRADE_TYPE_DESC


## 3 Reconciliação do cruzamento

Compara a quantidade de registros e a soma das vendas antes e depois da integração. Diferenças indicam possível perda ou multiplicação de registros; a cobertura deve ser conferida separadamente.

In [0]:
joined = sales.join(
    channels.select("channel_key", "TRADE_GROUP_DESC", "TRADE_TYPE_DESC"),
    ["channel_key"], "left",
)


def reconciliation(frame, stage):
    return frame.agg(
        F.count("*").alias("registros"),
        F.count("sales_amount").alias("valores_convertidos"),
        F.sum("sales_amount").alias("soma_vendas"),
    ).withColumn("etapa", F.lit(stage)).select(
        "etapa", "registros", "valores_convertidos", "soma_vendas"
    )


display(reconciliation(sales, "antes_join").unionByName(
    reconciliation(joined, "depois_left_join")
))
display(joined.select(
    "TRADE_CHNL_DESC", "channel_key", "TRADE_GROUP_DESC", "TRADE_TYPE_DESC", "$ Volume"
).limit(20))

etapa,registros,valores_convertidos,soma_vendas
antes_join,16151,16151,4348925.160000
depois_left_join,16151,16151,4348925.160000


TRADE_CHNL_DESC,channel_key,TRADE_GROUP_DESC,TRADE_TYPE_DESC,$ Volume
SPORT VENUE,SPORT VENUE,ENTERTAINMENT,ALCOHOLIC,22.48
SUPERETTE,SUPERETTE,SERVICES,MIX,100
PLANT / OFFICE,PLANT / OFFICE,SERVICES,MIX,66.14
MASS MERCHANDISER,MASS MERCHANDISER,GROCERY,MIX,222.5
MASS MERCHANDISER,MASS MERCHANDISER,GROCERY,MIX,302.5
LIQUOR/BEER/WINE/SOFT,LIQUOR/BEER/WINE/SOFT,GROCERY,ALCOHOLIC,10
CONVENIENCE STORE,CONVENIENCE STORE,SERVICES,MIX,-4.22
QUICK SERVICE RESTAURANT,QUICK SERVICE RESTAURANT,ENTERTAINMENT,ALCOHOLIC,59.95
SUPERMARKET,SUPERMARKET,GROCERY,MIX,300
QUICK SERVICE RESTAURANT,QUICK SERVICE RESTAURANT,ENTERTAINMENT,ALCOHOLIC,85


## 4 Relações entre códigos e descrições

Verifica quais atributos são determinados por cada código e quais apresentam variações. Os resultados orientam a definição das dimensões de produto e canal.

In [0]:
trimmed = sales_raw.select(*[F.trim(F.col(name)).alias(name) for name in sales_raw.columns])
relationships = [
    ("CE_BRAND_FLVR", "BRAND_NM"),
    ("CE_BRAND_FLVR", "PKG_CAT"),
    ("CE_BRAND_FLVR", "TSR_PCKG_NM"),
    ("PKG_CAT", "Pkg_Cat_Desc"),
    ("TRADE_CHNL_DESC", "CHNL_GROUP"),
]
for key, attribute in relationships:
    print(f"Relação investigada: {key} -> {attribute}")
    display(
        trimmed.groupBy(key).agg(
            F.countDistinct(attribute).alias("valores_distintos"),
            F.count(F.when(F.col(attribute).isNull() | (F.col(attribute) == ""), 1))
            .alias("atributos_ausentes"),
            F.sort_array(F.collect_set(attribute)).alias("valores_observados"),
        ).orderBy(F.desc("valores_distintos"), key).limit(50)
    )

Relação investigada: CE_BRAND_FLVR -> BRAND_NM


CE_BRAND_FLVR,valores_distintos,atributos_ausentes,valores_observados
3440,1,0,List(LEMON)
3441,1,0,List(RASPBERRY)
3554,1,0,List(STRAWBERRY)
3697,1,0,List(GRAPE)


Relação investigada: CE_BRAND_FLVR -> PKG_CAT


CE_BRAND_FLVR,valores_distintos,atributos_ausentes,valores_observados
3440,3,0,"List(N128, N20O, N56P)"
3441,2,0,"List(N20O, N56P)"
3554,2,0,"List(N20O, N56P)"
3697,1,0,List(N20O)


Relação investigada: CE_BRAND_FLVR -> TSR_PCKG_NM


CE_BRAND_FLVR,valores_distintos,atributos_ausentes,valores_observados
3440,8,0,"List(.591L NRP 24L, .591L NRP 24L *, .5L NRP 6P, .5L NRP 6P S, .5L NRP 6P*, 12Z NRP 8P F, 20Z NRP 24L, 20z NRP 24L S)"
3441,7,0,"List(.591L NRP 24L, .591L NRP 24L *, .5L NRP 6P, .5L NRP 6P S, .5L NRP 6P*, 20Z NRP 24L, 20z NRP 24L S)"
3554,6,0,"List(.591L NRP 24L, .591L NRP 24L *, .5L NRP 6P, .5L NRP 6P S, 20Z NRP 24L, 20z NRP 24L S)"
3697,1,0,List(20z NRP 24L S)


Relação investigada: PKG_CAT -> Pkg_Cat_Desc


PKG_CAT,valores_distintos,atributos_ausentes,valores_observados
N128,1,0,List(12Z/355M 8NR)
N20O,1,0,List(20Z/600ML)
N56P,1,0,List(500ML 6P)


Relação investigada: TRADE_CHNL_DESC -> CHNL_GROUP


TRADE_CHNL_DESC,valores_distintos,atributos_ausentes,valores_observados
ALL OTHER,1,0,List(OTHER SMALL STORES)
AMUSEMENT/RECREATIONAL,1,0,List(LEISURE)
AUTOMOTIVE SERVICES,1,0,List(RETAIL COLD)
BAR/TAVERN/LICENSED EST,1,0,List(FOOD SERVICE)
CASH + CARRY RETAIL,1,0,List(CLUBS)
CLUB STORE,1,0,List(CLUBS)
COLLEGE/UNIVERSITY,1,0,List(EDUCATION)
CONVENIENCE STORE,1,0,List(CONVENIENCE RETAIL)
DRUG STORE,1,0,List(DRUG STORES)
GENERAL MERCHANDISER,1,0,List(RETAIL COLD)


## 5 Granularidade das vendas

Avalia a multiplicidade da combinação candidata de data, produto, região, canal e embalagem. A unicidade observada descreve a amostra, mas não comprova uma chave transacional para futuras cargas.

In [0]:
candidate_grain = [
    "DATE", "CE_BRAND_FLVR", "BRAND_NM", "Btlr_Org_LVL_C_Desc", "CHNL_GROUP",
    "TRADE_CHNL_DESC", "PKG_CAT", "Pkg_Cat_Desc", "TSR_PCKG_NM",
]
grain_groups = trimmed.groupBy(*candidate_grain).agg(
    F.count("*").alias("registros"),
    F.countDistinct("$ Volume").alias("valores_venda_distintos"),
)
display(grain_groups.agg(
    F.count("*").alias("combinacoes"),
    F.sum("registros").alias("linhas"),
    F.count(F.when(F.col("registros") > 1, 1)).alias("combinacoes_repetidas"),
    F.max("registros").alias("maior_multiplicidade"),
))
display(grain_groups.filter(F.col("registros") > 1).orderBy(F.desc("registros")).limit(30))

combinacoes,linhas,combinacoes_repetidas,maior_multiplicidade
16151,16151,0,1


DATE,CE_BRAND_FLVR,BRAND_NM,Btlr_Org_LVL_C_Desc,CHNL_GROUP,TRADE_CHNL_DESC,PKG_CAT,Pkg_Cat_Desc,TSR_PCKG_NM,registros,valores_venda_distintos


## Leitura dos resultados

Conferir a cobertura dos canais, a conservação dos totais e as dependências entre atributos antes de definir as chaves do modelo.

As justificativas técnicas e as propostas de modelagem estão em [README](README.md).